<a href="https://colab.research.google.com/github/predpoke/SuperSecondProject/blob/KAN-4-image-analysis-model/SuperCaptionModel2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

from google.colab import drive
drive.mount('/content/drive')


PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

for path in [PROJECT_DIR, DATA_DIR, FAISS_DIR, MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
  os.makedirs(path, exist_ok=True)
print("프로젝트 폴더 생성 완료")
print(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
프로젝트 폴더 생성 완료
/content/drive/MyDrive/officehome_caption_project


In [4]:


cnn_checkpoint_path = "/content/drive/MyDrive/models/officehome_convnext_phase_center_best.pt"

print("checkpoint 존재:", os.path.exists(cnn_checkpoint_path))
print("checkpoint 경로:", cnn_checkpoint_path)

checkpoint 존재: True
checkpoint 경로: /content/drive/MyDrive/models/officehome_convnext_phase_center_best.pt


In [5]:
#Drive 폴더로 캡션 복사


from google.colab import files
uploaded = files.upload()

import shutil
src = "/content/office_home_image_caption_cleaned_metadata.csv"
dst_dir = "/content/drive/MyDrive/officehome_caption_project/data"
dst = f"{dst_dir}/office_home_image_caption_cleaned_metadata.csv"

shutil.copy(src, dst)

Saving office_home_image_caption_cleaned_metadata.csv to office_home_image_caption_cleaned_metadata (1).csv


'/content/drive/MyDrive/officehome_caption_project/data/office_home_image_caption_cleaned_metadata.csv'

In [6]:
csv_path = "/content/drive/MyDrive/officehome_caption_project/data/office_home_enriched_metadata.csv"

In [7]:
!pip install -q sentence-transformers faiss-cpu
import pandas as pd

df = pd.read_csv(csv_path)

df.head()

,dataset,row_idx,source_filepath,image_url,domain,object,price_usd,size,caption
0,Voxel51/Office-Home,0,data/data_0/00001.jpg,https://huggingface.co/datasets/Voxel51/Office...,Real World,Alarm_Clock,53.63,standard,A catalog caption identifies an Alarm Clock in...
1,Voxel51/Office-Home,1,data/data_0/00002.jpg,https://huggingface.co/datasets/Voxel51/Office...,Real World,Alarm_Clock,477.95,large,This Alarm Clock reference uses a real-world p...
2,Voxel51/Office-Home,2,data/data_0/00003.jpg,https://huggingface.co/datasets/Voxel51/Office...,Real World,Alarm_Clock,161.73,XS,"The Real World image contains an Alarm Clock, ..."
3,Voxel51/Office-Home,3,data/data_0/00004.jpg,https://huggingface.co/datasets/Voxel51/Office...,Real World,Alarm_Clock,366.93,oversized,A category-focused real-world photograph of an...
4,Voxel51/Office-Home,4,data/data_0/00005.jpg,https://huggingface.co/datasets/Voxel51/Office...,Real World,Alarm_Clock,137.78,M,An Alarm Clock in a real-world photograph is c...


In [8]:
#검색용 문장 만들기

def make_search_text(row):
  object_name = str(row.get("object", "")).replace("_", " ")
  domain = str(row.get("domain", ""))
  size = str(row.get("size", ""))
  price = str(row.get("price_usd", ""))
  caption = str(row.get("caption", ""))


  text = (
      f"object : {object_name} . "
      f"Domain : {domain}."
      f"Size : {size}"
      f"Price : {price} USD."
      f"Caption : {caption}. "
  )

  return text


df["search_text"] = df.apply(make_search_text, axis =1)
df[["object", "caption", "search_text"]].head()


,object,caption,search_text
0,Alarm_Clock,A catalog caption identifies an Alarm Clock in...,object : Alarm Clock . Domain : Real World.Siz...
1,Alarm_Clock,This Alarm Clock reference uses a real-world p...,object : Alarm Clock . Domain : Real World.Siz...
2,Alarm_Clock,"The Real World image contains an Alarm Clock, ...",object : Alarm Clock . Domain : Real World.Siz...
3,Alarm_Clock,A category-focused real-world photograph of an...,object : Alarm Clock . Domain : Real World.Siz...
4,Alarm_Clock,An Alarm Clock in a real-world photograph is c...,object : Alarm Clock . Domain : Real World.Siz...


In [9]:
#


from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(embedding_model_name)

texts = df["search_text"].fillna("").tolist()

embeddings= model.encode(
    texts,
    batch_size = 64,
    show_progress_bar = True,
    convert_to_tensor = True,
    normalize_embeddings=True
)

print("임베딩 shape", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/244 [00:00<?, ?it/s]

임베딩 shape torch.Size([15588, 384])


In [10]:
!pip install -q faiss-cpu

import faiss
import numpy as np
import torch

if isinstance(embeddings, torch.Tensor):
  embeddings_np = embeddings.detach().cpu().numpy()
else :
  embeddings_np = embeddings

embeddings_np = embeddings_np.astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [11]:
import json

faiss_index_path = os.path.join(FAISS_DIR , "officehome_caption.index")
metadata_path = os.path.join(DATA_DIR, "officehome_search_metadata.csv")
config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

faiss.write_index(index, faiss_index_path)

df.to_csv(metadata_path, index=False)

config = {
    "embedding_model_name": embedding_model_name,
    "embedding_dim ": int(dimension),
    "num_vectors" : int(index.ntotal),
    "metadata_path" : metadata_path,
    "faiss_index_path" : faiss_index_path
}

with open(config_path, "w", encoding="utf-8") as f:
  json.dump(config, f, ensure_ascii=False, indent=2)


In [12]:
def search_caption(query, top_k =5):
  query_vec = model.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  ).astype("float32")


  scores, indices = index.search(query_vec, top_k)


  results = df.iloc[indices[0]].copy()
  results["score"] = scores[0]

  cols = ["score", "object", "size", "price_usd", "caption", "image_url", "source_filepath"]
  available_cols = [c for c in cols if c in results. columns]

  return results[available_cols]


search_caption("black moniter with a stand", top_k =5)

,score,object,size,price_usd,caption,image_url,source_filepath
13550,0.431705,Monitor,oversized,495.23,This clip-art style illustration presents a Mo...,https://huggingface.co/datasets/Voxel51/Office...,data/data_67/00031-199.jpg
2405,0.430854,Monitor,compact,15.33,This real-world photograph presents a Monitor ...,https://huggingface.co/datasets/Voxel51/Office...,data/data_12/00025-34.jpg
6836,0.430760,Monitor,oversized,453.03,"This product listing photo shows a Monitor, wi...",https://huggingface.co/datasets/Voxel51/Office...,data/data_34/00060-70.jpg
10322,0.421337,Monitor,XL,253.95,"The Art image contains a Monitor, captured wit...",https://huggingface.co/datasets/Voxel51/Office...,data/data_51/00023-157.jpg
10316,0.419330,Monitor,compact,85.69,"This artistic rendering shows a Monitor, with ...",https://huggingface.co/datasets/Voxel51/Office...,data/data_51/00017-164.jpg


In [13]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)




In [14]:
print("전체 데이터 수:", len(df))
print("컬럼 목록:", df.columns.tolist())

print("\ncaption 결측치 개수:", df["caption"].isna().sum())
print("caption 빈 문자열 개수:", (df["caption"].fillna("").str.strip() == "").sum())

df[["object", "size", "price_usd", "caption"]].head(10)

전체 데이터 수: 15588
컬럼 목록: ['dataset', 'row_idx', 'source_filepath', 'image_url', 'domain', 'object', 'price_usd', 'size', 'caption', 'search_text']

caption 결측치 개수: 0
caption 빈 문자열 개수: 0


,object,size,price_usd,caption
0,Alarm_Clock,standard,53.63,"A catalog caption identifies an Alarm Clock in a real-world photograph, captured with a simple front-facing composition, against a plain or low-detail background, and useful for a marketplace-style item listing."
1,Alarm_Clock,large,477.95,"This Alarm Clock reference uses a real-world photograph; it is captured with a simple front-facing composition, with a background that does not dominate the item, and ready for visual search, tagging, or inventory matching."
2,Alarm_Clock,XS,161.73,"The Real World image contains an Alarm Clock, captured with a simple front-facing composition, using a practical reference-image style, and suited to a searchable product dataset."
3,Alarm_Clock,oversized,366.93,"A category-focused real-world photograph of an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and designed for object-aware metadata enrichment."
4,Alarm_Clock,M,137.78,"An Alarm Clock in a real-world photograph is captured with a simple front-facing composition, against a plain or low-detail background, and suitable for an office supply or home goods catalog."
5,Alarm_Clock,XL,313.96,"This real-world photograph presents an Alarm Clock as the primary object, captured with a simple front-facing composition, with a background that does not dominate the item, and appropriate for product discovery and category browsing."
6,Alarm_Clock,XL,210.43,"A labeled Alarm Clock image uses a real-world photograph; it is captured with a simple front-facing composition, using a practical reference-image style, and written for a compact retail caption field."
7,Alarm_Clock,oversized,493.68,"In the Real World domain, an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and prepared for lightweight catalog annotation."
8,Alarm_Clock,oversized,269.10,"Office-Home presents an Alarm Clock as a real-world photograph; it is captured with a simple front-facing composition, against a plain or low-detail background, and ready for visual search, tagging, or inventory matching."
9,Alarm_Clock,standard,179.47,"A real-world photograph of an Alarm Clock, with a background that does not dominate the item; it is captured with a simple front-facing composition and prepared as synthetic commerce metadata."


In [15]:
## 여기서부터 gradio 작업 시작



!pip install -q gradio


PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
FAISS_DIR =os.path.join(PROJECT_DIR, 'faiss_index')

faiss_index_path = os.path.join(FAISS_DIR, "officehome_caption.index")
metadata_path = os.path.join(DATA_DIR, "officehome_search_metadata.csv")
config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

#config 불러오기
with open(config_path, "r", encoding="utf-8") as f:
  config = json.load(f)

embedding_model_name = config["embedding_model_name"]


#모델, metadata, FAISS index 불러오기

model = SentenceTransformer(embedding_model_name)
df = pd.read_csv(metadata_path)
index = faiss.read_index(faiss_index_path)


#혹시 이름 바꾸면 이거 쓰기
# faiss_index_path = os.path.join(FAISS_DIR, "네가_저장한_index_파일명.index")
# metadata_path = os.path.join(DATA_DIR, "네가_저장한_metadata_파일명.csv")
# config_path = os.path.join(FAISS_DIR, "네가_저장한_config_파일명.json")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
print(config)
print(config.keys())

{'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim ': 384, 'num_vectors': 15588, 'metadata_path': '/content/drive/MyDrive/officehome_caption_project/data/officehome_search_metadata.csv', 'faiss_index_path': '/content/drive/MyDrive/officehome_caption_project/faiss_index/officehome_caption.index'}
dict_keys(['embedding_model_name', 'embedding_dim ', 'num_vectors', 'metadata_path', 'faiss_index_path'])


In [17]:
def search_caption_gradio(query, top_k) :
  if query is None or query.strip() == "":
    return pd.DataFrame({"message" : ["검색어를 입력해주세요."]})

  top_k = int(top_k)

  query_vec = model.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  ).astype("float32")

  scores, indices = index.search(query_vec, top_k)

  results = df.iloc[indices[0]].copy()
  results["score"] = scores[0]

  cols =[
      "score",
      "object",
      "domain",
      "size",
      "price_usd",
      "caption",
      "image_url",
      "source_filepath"
  ]


  available_cols = [c for c in cols if c in results.columns]

  results = results[available_cols]


  #score 반올림 보기 좋으라고
  if "score" in results.columns :
    results["score"] = results["score"].round(4)

    return results



In [18]:
search_caption_gradio("black moniter with a stand ", 5)


,score,object,domain,size,price_usd,caption,image_url,source_filepath
13550,0.4317,Monitor,Clipart,oversized,495.23,"This clip-art style illustration presents a Monitor as the primary object, presented with emphasis on the item silhouette, with a background that does not dominate the item, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_67/00031-199.jpg,data/data_67/00031-199.jpg
2405,0.4309,Monitor,Real World,compact,15.33,"This real-world photograph presents a Monitor as the primary object, captured with a simple front-facing composition, with a composition suitable for search indexing, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_12/00025-34.jpg,data/data_12/00025-34.jpg
6836,0.4308,Monitor,Product,oversized,453.03,"This product listing photo shows a Monitor, with a compact scene around the object; the item is presented with emphasis on the item silhouette and written for a compact retail caption field.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_34/00060-70.jpg,data/data_34/00060-70.jpg
10322,0.4213,Monitor,Art,XL,253.95,"The Art image contains a Monitor, captured with a simple front-facing composition, with a composition suitable for search indexing, and suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00023-157.jpg,data/data_51/00023-157.jpg
10316,0.4193,Monitor,Art,compact,85.69,"This artistic rendering shows a Monitor, with the item separated from visual distractions; the item is presented as the main subject and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00017-164.jpg,data/data_51/00017-164.jpg


In [19]:
import gradio as gr

with gr.Blocks(title="OfficeHome Caption Search") as demo :
  gr.Markdown(
      """
      자연어 검색어를 입력하면,
      SentenceTransformer + FAISS를 이용해 가장 유사한 이미지 캡션 데이털르 검색합니다
      """
  )


  with gr.Row():
    query_input = gr.Textbox(
        label="검색어",
        placeholder="예 : black moniter with a stand / 투명한 물병 / small alarm"
    )

    top_k_input = gr.Slider(
        minimum=1,
        maximum=20,
        value=5,
        step=1,
        label="검색 결과 개수"
    )

  search_button = gr.Button("검색하기")

  result_table = gr.Dataframe(
     label = "검색결과",
     wrap=True
 )

  search_button.click(
      fn =search_caption_gradio,
      inputs=[query_input, top_k_input],
      outputs=result_table
)


demo.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2af2d7328bc90892a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:


import gradio as gr

with gr.Blocks(title="OfficeHome Caption Search") as demo:
    gr.Markdown(
        """
        # OfficeHome Caption Search

        자연어 검색어를 입력하면,
        FAISS가 가장 유사한 이미지 캡션 데이터를 찾아 이미지 카드로 보여줍니다.
        """
    )

    with gr.Row():
        query_input = gr.Textbox(
            label="검색어",
            placeholder="예: black monitor with a stand / 투명한 물병 / small alarm clock",
            lines=2
        )

        top_k_input = gr.Slider(
            minimum=1,
            maximum=12,
            value=5,
            step=1,
            label="검색 결과 개수"
        )

    search_button = gr.Button("검색하기")

    result_html = gr.HTML(label="검색 결과")

    search_button.click(
        fn=search_caption_gradio,
        inputs=[query_input, top_k_input],
        outputs=result_html
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d84907220b491d85a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
test_queries = [
    "real world alarm clock",
    "catalog photo of an alarm clock",
    "black monitor with a stand",
    "product photo of a monitor",
    "bottle for water",
    "office chair product image",
    "yellow pencil for writing",
    "computer keyboard on a desk"
]

for q in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", q)
    display(search_caption_gradio(q, 5))


QUERY: real world alarm clock


,score,object,domain,size,price_usd,caption,image_url,source_filepath
28,0.7782,Alarm_Clock,Real World,compact,325.52,"This Office-Home Real World sample contains an Alarm Clock, centered in the frame, against a plain or low-detail background, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00029.jpg,data/data_0/00029.jpg
7,0.7735,Alarm_Clock,Real World,oversized,493.68,"In the Real World domain, an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and prepared for lightweight catalog annotation.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00008.jpg,data/data_0/00008.jpg
58,0.7709,Alarm_Clock,Real World,XL,169.21,"This Office-Home Real World sample contains an Alarm Clock, displayed in a practical inventory-style view, with the item separated from visual distractions, and prepared for lightweight catalog annotation.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00059.jpg,data/data_0/00059.jpg
13,0.7697,Alarm_Clock,Real World,standard,149.83,"This Office-Home Real World sample contains an Alarm Clock, shown from a straightforward angle, with a background that does not dominate the item, and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00014.jpg,data/data_0/00014.jpg
67,0.7616,Alarm_Clock,Real World,S,448.58,"In the Real World domain, an Alarm Clock is displayed in a practical inventory-style view, in a simple office-home dataset scene, and prepared as synthetic commerce metadata.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00068.jpg,data/data_0/00068.jpg



QUERY: catalog photo of an alarm clock


,score,object,domain,size,price_usd,caption,image_url,source_filepath
4416,0.7799,Alarm_Clock,Product,XL,441.31,"A labeled Alarm Clock image uses a product listing photo; it is shown as a standalone office-home item, with the item separated from visual distractions, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_22/00060-47.jpg,data/data_22/00060-47.jpg
8822,0.7752,Alarm_Clock,Art,large,303.43,"The Art image contains an Alarm Clock, positioned clearly for catalog review, with the item separated from visual distractions, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_44/00027-130.jpg,data/data_44/00027-130.jpg
4431,0.7751,Alarm_Clock,Product,XL,332.03,"A labeled Alarm Clock image uses a product listing photo; it is presented with emphasis on the item silhouette, in a straightforward visual sample, and appropriate for product discovery and category browsing.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_22/00075-24.jpg,data/data_22/00075-24.jpg
18,0.7692,Alarm_Clock,Real World,compact,14.27,"A category-focused real-world photograph of an Alarm Clock is shown from a straightforward angle, using a practical reference-image style, and prepared for lightweight catalog annotation.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00019.jpg,data/data_0/00019.jpg
8852,0.7611,Alarm_Clock,Art,S,387.25,"The Art image contains an Alarm Clock, framed for quick visual identification, with a composition suitable for search indexing, and prepared for lightweight catalog annotation.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_44/00057-95.jpg,data/data_44/00057-95.jpg



QUERY: black monitor with a stand


...

,score,object,domain,size,price_usd,caption,image_url,source_filepath
5042,0.7870,Chair,Product,large,403.24,"The Product image contains a Chair, captured with a simple front-facing composition, with a background that does not dominate the item, and appropriate for product discovery and category browsing.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_25/00032-74.jpg,data/data_25/00032-74.jpg
5057,0.7812,Chair,Product,compact,297.04,"The Product image contains a Chair, shown from a straightforward angle, against a plain or low-detail background, and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_25/00047-68.jpg,data/data_25/00047-68.jpg
5079,0.7631,Chair,Product,XL,57.06,"A product listing photo of a Chair, using a practical reference-image style; it is centered in the frame and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_25/00069-32.jpg,data/data_25/00069-32.jpg
9373,0.7613,Chair,Art,S,57.59,"This Office-Home Art sample contains a Chair, shown from a straightforward angle, using a practical reference-image style, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_46/00053-105.jpg,data/data_46/00053-105.jpg
5023,0.7588,Chair,Product,XL,336.14,"This Office-Home Product sample contains a Chair, presented as the main subject, using a practical reference-image style, and suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_25/00013-76.jpg,data/data_25/00013-76.jpg



QUERY: yellow pencil for writing


,score,object,domain,size,price_usd,caption,image_url,source_filepath
10548,0.5323,Pencil,Art,S,85.35,"A category-focused artistic rendering of a Pencil is presented as the main subject, in a clean classification-friendly layout, and appropriate for product discovery and category browsing.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_52/00009-174.jpg,data/data_52/00009-174.jpg
10540,0.5303,Pencil,Art,oversized,390.33,"A Pencil is framed for quick visual identification in a artistic rendering, in a clean classification-friendly layout, and is suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_52/00001-174.jpg,data/data_52/00001-174.jpg
10564,0.5229,Pencil,Art,M,241.76,"A Pencil in a artistic rendering is captured with a simple front-facing composition, in a clean classification-friendly layout, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_52/00025-158.jpg,data/data_52/00025-158.jpg
7449,0.5175,Pencil,Product,S,72.18,"A product listing photo of a Pencil, in a simple office-home dataset scene; it is captured with a simple front-facing composition and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_37/00028-108.jpg,data/data_37/00028-108.jpg
2920,0.5165,Pencil,Real World,standard,96.02,"A Pencil is centered in the frame in a real-world photograph, against a plain or low-detail background, and is prepared for lightweight catalog annotation.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_14/00010-44.jpg,data/data_14/00010-44.jpg



QUERY: computer keyboard on a desk


,score,object,domain,size,price_usd,caption,image_url,source_filepath
6474,0.6371,Keyboard,Product,large,57.64,"A product listing photo of a Keyboard, with the surrounding area kept secondary; it is presented as the main subject and suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_32/00047-85.jpg,data/data_32/00047-85.jpg
6504,0.6012,Keyboard,Product,XL,462.38,"A product listing photo of a Keyboard, with a background that does not dominate the item; it is shown from a straightforward angle and appropriate for product discovery and category browsing.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_32/00077-32.jpg,data/data_32/00077-32.jpg
6478,0.5984,Keyboard,Product,compact,148.55,"This Office-Home Product sample contains a Keyboard, captured with a simple front-facing composition, with the surrounding area kept secondary, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_32/00051-83.jpg,data/data_32/00051-83.jpg
6508,0.5971,Keyboard,Product,oversized,434.60,"This Office-Home Product sample contains a Keyboard, centered in the frame, with a background that does not dominate the item, and prepared as synthetic commerce metadata.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_32/00081-26.jpg,data/data_32/00081-26.jpg
6493,0.5922,Keyboard,Product,XL,346.54,"This Office-Home Product sample contains a Keyboard, shown from a straightforward angle, using a practical reference-image style, and appropriate for product discovery and category browsing.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_32/00066-54.jpg,data/data_32/00066-54.jpg


In [22]:
import os
import json

PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")

metadata_path = os.path.join(DATA_DIR, "officehome_search_metadata.csv")
faiss_index_path = os.path.join(FAISS_DIR, "officehome_caption.index")
config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

config = {
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "metadata_path": metadata_path,
    "faiss_index_path": faiss_index_path,
    "index_type": "IndexFlatIP",
    "normalize_embeddings": True,
    "description": "OfficeHome caption metadata FAISS search index"
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("caption config 저장 완료")
print(config)

caption config 저장 완료
{'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'metadata_path': '/content/drive/MyDrive/officehome_caption_project/data/officehome_search_metadata.csv', 'faiss_index_path': '/content/drive/MyDrive/officehome_caption_project/faiss_index/officehome_caption.index', 'index_type': 'IndexFlatIP', 'normalize_embeddings': True, 'description': 'OfficeHome caption metadata FAISS search index'}


In [23]:
import os
import json
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

CAPTION_SEARCH_LOADED = False

def load_caption_search_once():
    global CAPTION_SEARCH_LOADED
    global caption_model, caption_df, caption_index

    if CAPTION_SEARCH_LOADED:
        print("Caption search는 이미 로드되어 있습니다.")
        return caption_model, caption_df, caption_index

    PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"
    FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")

    config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    embedding_model_name = config["embedding_model_name"]
    metadata_path = config["metadata_path"]
    faiss_index_path = config["faiss_index_path"]

    caption_model = SentenceTransformer(embedding_model_name)
    caption_df = pd.read_csv(metadata_path)
    caption_index = faiss.read_index(faiss_index_path)

    CAPTION_SEARCH_LOADED = True

    print("Caption search 로드 완료")
    print("Embedding model:", embedding_model_name)
    print("Metadata shape:", caption_df.shape)
    print("FAISS vector count:", caption_index.ntotal)

    return caption_model, caption_df, caption_index

In [24]:
def search_caption_gradio(query, top_k=5):
    if query is None or query.strip() == "":
        return pd.DataFrame({"message": ["검색어를 입력해주세요."]})

    top_k = int(top_k)

    query_vec = caption_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = caption_index.search(query_vec, top_k)

    results = caption_df.iloc[indices[0]].copy()
    results["score"] = scores[0]

    cols = [
        "score",
        "object",
        "domain",
        "size",
        "price_usd",
        "caption",
        "image_url",
        "source_filepath"
    ]

    available_cols = [c for c in cols if c in results.columns]
    results = results[available_cols]

    if "score" in results.columns:
        results["score"] = results["score"].round(4)

    return results

In [25]:
import io
import base64
import torch
import torch.nn.functional as F
from PIL import Image

# ---------------------------------------------------
# 1. CNN으로 업로드 이미지 분류
#    (cnn_model, cnn_val_transform, classes, device 가 이미 로드되어 있다고 가정)
# ---------------------------------------------------
def predict_image_class(pil_image):
    if pil_image is None:
        return None, None

    image = pil_image.convert("RGB")
    image_tensor = cnn_val_transform(image).unsqueeze(0).to(device)

    cnn_model.eval()
    with torch.no_grad():
        logits = cnn_model(image_tensor)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = torch.max(probs, dim=1)

    pred_class = classes[pred_idx.item()]
    confidence = float(conf.item())

    return pred_class, confidence


# ---------------------------------------------------
# 2. PIL 이미지를 HTML에 표시할 수 있게 base64 변환
# ---------------------------------------------------
def pil_to_base64(pil_image, max_size=(220, 220)):
    img = pil_image.copy()
    img.thumbnail(max_size)

    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


# ---------------------------------------------------
# 3. 검색 결과를 HTML 카드 형태로 변환
# ---------------------------------------------------
def dataframe_to_result_html(results_df, title="검색 결과", extra_info=""):
    if results_df is None or len(results_df) == 0:
        return f"""
        <div style="padding:16px; border:1px solid #ddd; border-radius:12px;">
            <h3>{title}</h3>
            <p>검색 결과가 없습니다.</p>
            <div>{extra_info}</div>
        </div>
        """

    html = f"""
    <div style="padding:16px;">
        <h3>{title}</h3>
        <div style="margin-bottom:12px;">{extra_info}</div>
    """

    for _, row in results_df.iterrows():
        image_url = row["image_url"] if "image_url" in row else ""
        score = row["score"] if "score" in row else ""
        obj = row["object"] if "object" in row else ""
        domain = row["domain"] if "domain" in row else ""
        size = row["size"] if "size" in row else ""
        price_usd = row["price_usd"] if "price_usd" in row else ""
        caption = row["caption"] if "caption" in row else ""
        source_filepath = row["source_filepath"] if "source_filepath" in row else ""

        html += f"""
        <div style="
            border:1px solid #ddd;
            border-radius:14px;
            padding:14px;
            margin-bottom:14px;
            display:flex;
            gap:16px;
            align-items:flex-start;
            background:#fafafa;
        ">
            <div style="min-width:180px;">
                <img src="{image_url}" style="width:180px; border-radius:10px; border:1px solid #eee;" />
            </div>
            <div style="flex:1;">
                <p><b>score</b>: {score:.4f}</p>
                <p><b>object</b>: {obj}</p>
                <p><b>domain</b>: {domain}</p>
                <p><b>size</b>: {size}</p>
                <p><b>price_usd</b>: {price_usd}</p>
                <p><b>caption</b>: {caption}</p>
                <p><b>source</b>: {source_filepath}</p>
            </div>
        </div>
        """

    html += "</div>"
    return html


# ---------------------------------------------------
# 4. 기존 텍스트 검색용 Gradio 함수
#    (search_caption(query, top_k)는 이미 만들어져 있다고 가정)
# ---------------------------------------------------
def search_caption_text_gradio(query, top_k):
    if query is None or str(query).strip() == "":
        return """
        <div style="padding:16px; border:1px solid #ddd; border-radius:12px;">
            검색어를 입력해주세요.
        </div>
        """

    top_k = int(top_k)
    results = search_caption(query, top_k=top_k)

    extra_info = f"""
    <div style="padding:10px; background:#f3f4f6; border-radius:10px;">
        <b>입력 검색어:</b> {query}
    </div>
    """

    return dataframe_to_result_html(results, title="텍스트 검색 결과", extra_info=extra_info)


# ---------------------------------------------------
# 5. 이미지 + 텍스트 결합 검색용 함수
# ---------------------------------------------------
def search_image_and_caption_gradio(image, query, top_k):
    if image is None and (query is None or str(query).strip() == ""):
        return """
        <div style="padding:16px; border:1px solid #ddd; border-radius:12px;">
            이미지 또는 검색어 중 하나 이상 입력해주세요.
        </div>
        """

    pred_class, confidence = None, None
    image_html = ""

    # 1) 이미지가 있으면 CNN 분류
    if image is not None:
        pred_class, confidence = predict_image_class(image)
        img_b64 = pil_to_base64(image)

        image_html = f"""
        <div style="display:flex; gap:16px; align-items:flex-start; margin-bottom:16px;">
            <img src="{img_b64}" style="width:180px; border-radius:12px; border:1px solid #ddd;" />
            <div style="padding:10px; background:#eef6ff; border-radius:10px;">
                <p><b>CNN 예측 클래스:</b> {pred_class}</p>
                <p><b>예측 확률:</b> {confidence:.4f}</p>
            </div>
        </div>
        """

    # 2) 최종 검색어 만들기
    query = "" if query is None else str(query).strip()

    if pred_class is not None and query != "":
        final_query = f"{query}, {pred_class}"
    elif pred_class is not None:
        final_query = pred_class
    else:
        final_query = query

    # 3) FAISS 검색
    top_k = int(top_k)
    results = search_caption(final_query, top_k=top_k)

    extra_info = f"""
    {image_html}
    <div style="padding:10px; background:#f3f4f6; border-radius:10px;">
        <p><b>사용자 입력 문장:</b> {query if query != '' else '(없음)'}</p>
        <p><b>최종 검색어:</b> {final_query}</p>
    </div>
    """

    return dataframe_to_result_html(results, title="이미지 + 텍스트 검색 결과", extra_info=extra_info)

In [26]:
import gradio as gr

with gr.Blocks(title="OfficeHome Search System") as demo:
    gr.Markdown(
        """
        # OfficeHome Search System

        아래 두 가지 방식으로 검색할 수 있습니다.

        1. **텍스트 검색**
           - 자연어 검색어로 캡션/메타데이터 검색

        2. **이미지 + 텍스트 검색**
           - 이미지를 업로드하면 CNN이 먼저 분류하고,
           - 그 결과를 이용해 FAISS 캡션 검색을 수행
        """
    )

    with gr.Tabs():

        # ---------------------------------------------------
        # 탭 1: 텍스트 검색
        # ---------------------------------------------------
        with gr.Tab("텍스트 검색"):
            with gr.Row():
                text_query_input = gr.Textbox(
                    label="검색어",
                    placeholder="예: black monitor with a stand / transparent water bottle / small alarm clock",
                    lines=2
                )

                text_top_k_input = gr.Slider(
                    minimum=1,
                    maximum=12,
                    value=5,
                    step=1,
                    label="검색 결과 개수"
                )

            text_search_button = gr.Button("텍스트 검색하기")
            text_result_html = gr.HTML(label="텍스트 검색 결과")

            text_search_button.click(
                fn=search_caption_text_gradio,
                inputs=[text_query_input, text_top_k_input],
                outputs=text_result_html
            )

        # ---------------------------------------------------
        # 탭 2: 이미지 + 텍스트 검색
        # ---------------------------------------------------
        with gr.Tab("이미지 + 텍스트 검색"):
            with gr.Row():
                image_input = gr.Image(
                    type="pil",
                    label="이미지 업로드"
                )

                with gr.Column():
                    image_query_input = gr.Textbox(
                        label="추가 설명(선택)",
                        placeholder="예: black monitor with a stand / office product / screen only",
                        lines=2
                    )

                    image_top_k_input = gr.Slider(
                        minimum=1,
                        maximum=12,
                        value=5,
                        step=1,
                        label="검색 결과 개수"
                    )

                    image_search_button = gr.Button("이미지 + 텍스트 검색하기")

            image_result_html = gr.HTML(label="이미지 + 텍스트 검색 결과")

            image_search_button.click(
                fn=search_image_and_caption_gradio,
                inputs=[image_input, image_query_input, image_top_k_input],
                outputs=image_result_html
            )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80d83bb63a5ed2bea4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
PROJECT_DIR = "."

cnn_checkpoint_path = "/content/drive/MyDrive/models/officehome_convnext_phase_center_best.pt"

PROJECT_DIR = "officehome_caption_project"

metadata_path = f"{PROJECT_DIR}/officehome_search_metadata.csv"
index_path = f"{PROJECT_DIR}/officehome_caption.index"
config_path = f"{PROJECT_DIR}/officehome_caption_config.json"

cnn_checkpoint_path = "models/officehome_convnext_phase_center_best.pt"


import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_checkpoint_path = "/content/drive/MyDrive/models/officehome_convnext_phase_center_best.pt"

checkpoint = torch.load(cnn_checkpoint_path, map_location=device)

classes = checkpoint["classes"]
num_classes = len(classes)

cnn_model = models.convnext_tiny(weights=None)

in_features = cnn_model.classifier[2].in_features
cnn_model.classifier[2] = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, num_classes))

cnn_model.load_state_dict(checkpoint["model_state_dict"])
cnn_model = cnn_model.to(device)
cnn_model.eval()

cnn_val_transform = transforms.Compose([
    transforms.Resize((224, 384)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("CNN 모델 로드 완료")
print("클래스 수:", len(classes))
print("클래스 예시:", classes[:10])

CNN 모델 로드 완료
클래스 수: 65
클래스 예시: ['Alarm_Clock', 'Backpack', 'Batteries', 'Bed', 'Bike', 'Bottle', 'Bucket', 'Calculator', 'Calendar', 'Candles']


In [28]:
import gradio as gr

with gr.Blocks(title="OfficeHome CNN + Caption Search") as demo:
    gr.Markdown("""
    # OfficeHome CNN + Caption Search

    이미지를 업로드하면 CNN이 클래스를 예측하고,
    그 예측 결과를 Caption Search에 연결합니다.
    """)

    with gr.Tab("이미지 + 텍스트 검색"):
        image_input = gr.Image(
            type="pil",
            label="이미지 업로드"
        )

        image_query_input = gr.Textbox(
            label="추가 검색어",
            placeholder="예: black monitor, office desk item, transparent bottle",
            lines=2
        )

        image_top_k_input = gr.Slider(
            minimum=1,
            maximum=12,
            value=5,
            step=1,
            label="검색 결과 개수"
        )

        image_search_button = gr.Button("이미지 기반 검색하기")

        image_result_html = gr.HTML(
            label="이미지 + 텍스트 검색 결과"
        )

        image_search_button.click(
            fn=search_image_and_caption_gradio,
            inputs=[image_input, image_query_input, image_top_k_input],
            outputs=image_result_html
        )

    with gr.Tab("텍스트 검색"):
        text_query_input = gr.Textbox(
            label="검색어",
            placeholder="예: monitor, laptop, desk lamp, transparent bottle",
            lines=2
        )

        text_top_k_input = gr.Slider(
            minimum=1,
            maximum=12,
            value=5,
            step=1,
            label="검색 결과 개수"
        )

        text_search_button = gr.Button("텍스트 검색하기")

        text_result_html = gr.HTML(
            label="텍스트 검색 결과"
        )

        text_search_button.click(
            fn=search_caption_text_gradio,
            inputs=[text_query_input, text_top_k_input],
            outputs=text_result_html
        )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2bc79e9d7d80731dea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
import os
import shutil
import json

# =========================
# 1. Colab / Drive 원본 경로
# =========================

DRIVE_CAPTION_PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"

# 네가 실제로 찾은 CNN 모델 경로로 수정
DRIVE_CNN_MODEL_PATH = "/content/drive/MyDrive/models/officehome_convnext_phase_center_best.pt"

# =========================
# 2. Hugging Face 제출용 폴더
# =========================

HF_SPACE_DIR = "/content/officehome_hf_space"

HF_MODEL_DIR = os.path.join(HF_SPACE_DIR, "models")
HF_CAPTION_PROJECT_DIR = os.path.join(HF_SPACE_DIR, "officehome_caption_project")
HF_FAISS_DIR = os.path.join(HF_CAPTION_PROJECT_DIR, "faiss_index")

os.makedirs(HF_MODEL_DIR, exist_ok=True)
os.makedirs(HF_CAPTION_PROJECT_DIR, exist_ok=True)
os.makedirs(HF_FAISS_DIR, exist_ok=True)

# =========================
# 3. CNN 모델 복사
# =========================

shutil.copy2(
    DRIVE_CNN_MODEL_PATH,
    os.path.join(HF_MODEL_DIR, "officehome_convnext_phase_center_best.pt")
)

print("CNN 모델 복사 완료")

# =========================
# 4. Caption 프로젝트 파일 복사
# =========================

# faiss_index 폴더 전체 복사
drive_faiss_dir = os.path.join(DRIVE_CAPTION_PROJECT_DIR, "faiss_index")

for filename in os.listdir(drive_faiss_dir):
    src = os.path.join(drive_faiss_dir, filename)
    dst = os.path.join(HF_FAISS_DIR, filename)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("FAISS 폴더 복사 완료")

# metadata csv가 project 루트에 있는 경우 복사
for filename in os.listdir(DRIVE_CAPTION_PROJECT_DIR):
    src = os.path.join(DRIVE_CAPTION_PROJECT_DIR, filename)
    dst = os.path.join(HF_CAPTION_PROJECT_DIR, filename)

    if os.path.isfile(src) and filename.endswith(".csv"):
        shutil.copy2(src, dst)
        print("metadata 복사:", filename)

print("Hugging Face 제출용 폴더 생성 완료:", HF_SPACE_DIR)

CNN 모델 복사 완료
FAISS 폴더 복사 완료
Hugging Face 제출용 폴더 생성 완료: /content/officehome_hf_space


In [30]:
import os
import json

HF_SPACE_DIR = "/content/officehome_hf_space"

config_path = os.path.join(
    HF_SPACE_DIR,
    "officehome_caption_project",
    "faiss_index",
    "officehome_caption_config.json"
)

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print("수정 전 config:")
print(json.dumps(config, ensure_ascii=False, indent=2))

# =========================
# Hugging Face Space 기준 상대경로로 수정
# =========================

config["metadata_path"] = "officehome_caption_project/officehome_search_metadata.csv"
config["faiss_index_path"] = "officehome_caption_project/faiss_index/officehome_caption.index"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\n수정 후 config:")
print(json.dumps(config, ensure_ascii=False, indent=2))

requirements_text = """gradio
torch
torchvision
pandas
numpy
Pillow
sentence-transformers
faiss-cpu
"""

requirements_path = "/content/officehome_hf_space/requirements.txt"

with open(requirements_path, "w", encoding="utf-8") as f:
    f.write(requirements_text)

print("requirements.txt 생성 완료:", requirements_path)

수정 전 config:
{
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "metadata_path": "/content/drive/MyDrive/officehome_caption_project/data/officehome_search_metadata.csv",
  "faiss_index_path": "/content/drive/MyDrive/officehome_caption_project/faiss_index/officehome_caption.index",
  "index_type": "IndexFlatIP",
  "normalize_embeddings": true,
  "description": "OfficeHome caption metadata FAISS search index"
}

수정 후 config:
{
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "metadata_path": "officehome_caption_project/officehome_search_metadata.csv",
  "faiss_index_path": "officehome_caption_project/faiss_index/officehome_caption.index",
  "index_type": "IndexFlatIP",
  "normalize_embeddings": true,
  "description": "OfficeHome caption metadata FAISS search index"
}
requirements.txt 생성 완료: /content/officehome_hf_space/requirements.txt


In [31]:
import os
import json
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

import io
import base64
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import gradio as gr


# =========================
# Hugging Face Space 상대경로
# =========================

PROJECT_DIR = "officehome_caption_project"
FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")

config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

cnn_checkpoint_path = "models/officehome_convnext_phase_center_best.pt"

In [32]:
%cd /content/officehome_hf_space

/content/officehome_hf_space


In [33]:
CAPTION_SEARCH_LOADED = False

def load_caption_search_once():
    global CAPTION_SEARCH_LOADED
    global caption_model, caption_df, caption_index

    if CAPTION_SEARCH_LOADED:
        print("Caption search는 이미 로드되어 있습니다.")
        return caption_model, caption_df, caption_index

    PROJECT_DIR = "officehome_caption_project"
    FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")

    config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    embedding_model_name = config["embedding_model_name"]
    metadata_path = "/content/drive/MyDrive/officehome_caption_project/data/officehome_search_metadata.csv"
    faiss_index_path = config["faiss_index_path"]

    caption_model = SentenceTransformer(embedding_model_name)
    caption_df = pd.read_csv(metadata_path)
    caption_index = faiss.read_index(faiss_index_path)

    CAPTION_SEARCH_LOADED = True

    print("Caption search 로드 완료")
    print("Embedding model:", embedding_model_name)
    print("Metadata shape:", caption_df.shape)
    print("FAISS vector count:", caption_index.ntotal)

    return caption_model, caption_df, caption_index


caption_model, caption_df, caption_index = load_caption_search_once()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn_checkpoint_path = "models/officehome_convnext_phase_center_best.pt"

checkpoint = torch.load(cnn_checkpoint_path, map_location=device)

classes = checkpoint["classes"]
num_classes = len(classes)

cnn_model = models.convnext_tiny(weights=None)

in_features = cnn_model.classifier[2].in_features

cnn_model.classifier[2] = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, num_classes)
)

cnn_model.load_state_dict(checkpoint["model_state_dict"])
cnn_model = cnn_model.to(device)
cnn_model.eval()

cnn_val_transform = transforms.Compose([
    transforms.Resize((224, 384)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("CNN 모델 로드 완료")
print("클래스 수:", len(classes))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Caption search 로드 완료
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Metadata shape: (15588, 10)
FAISS vector count: 15588
CNN 모델 로드 완료
클래스 수: 65


In [34]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.lower().endswith(".csv"):
            if "metadata" in file.lower() or "caption" in file.lower() or "officehome" in file.lower():
                print(os.path.join(root, file))

/content/drive/MyDrive/datasets/HAM10000_plus_SkinCAP_metadata.csv
/content/drive/MyDrive/datasets/skincap_caption_labeled.csv
/content/drive/MyDrive/datasets/officehome_metadata.csv
/content/drive/MyDrive/datasets/officehome_train.csv
/content/drive/MyDrive/datasets/officehome_val.csv
/content/drive/MyDrive/datasets/officehome_train_local.csv
/content/drive/MyDrive/datasets/officehome_val_local.csv
/content/drive/MyDrive/datasets/officehome_metadata_local.csv
/content/drive/MyDrive/datasets/officehome_bad_images.csv
/content/drive/MyDrive/datasets/officehome_train_cnn_clean.csv
/content/drive/MyDrive/datasets/officehome_val_cnn_clean.csv
/content/drive/MyDrive/datasets/officehome_metadata_cnn_clean.csv
/content/drive/MyDrive/datasets/SkinCAP_HAM_format/skincap_ham_metadata.csv
/content/drive/MyDrive/officehome_caption_project/data/office_home_enriched_metadata.csv
/content/drive/MyDrive/officehome_caption_project/data/office_home_image_caption_cleaned_metadata.csv
/content/drive/MyDri

In [41]:
!pip install -U huggingface_hub

from huggingface_hub import login

login()


from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/officehome_hf_space",
    repo_id = "spsychic2/officehome-cnn-caption-search",
    repo_type = "space"
)



CommitInfo(commit_url='https://huggingface.co/spaces/spsychic2/officehome-cnn-caption-search/commit/c4ca460dd2a92e2434c713fb63368665a614c6f0', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c4ca460dd2a92e2434c713fb63368665a614c6f0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/spsychic2/officehome-cnn-caption-search', endpoint='https://huggingface.co', repo_type='space', repo_id='spsychic2/officehome-cnn-caption-search'), pr_revision=None, pr_num=None)